# Analyze Missing Names in Approved Inquiries
## Filter: Templates with 'Tel Aviv' or 'Israel'


In [4]:
# ============================================================================
# SETUP: Persona API Configuration
# ============================================================================

import os
import time
from typing import Dict, Any, List, Optional
import pandas as pd
import requests
from dotenv import load_dotenv

# Config
REQUEST_TIMEOUT = 30
SLEEP_S = 0.25  # be kind to API

load_dotenv()
PERSONA_API_KEY = os.getenv("PERSONA_API_KEY")

if not PERSONA_API_KEY:
    raise RuntimeError("Missing PERSONA_API_KEY in .env")

BASE_URL = "https://api.withpersona.com/api/v1"
HEADERS = {
    "Authorization": f"Bearer {PERSONA_API_KEY}",
    "Accept": "application/json",
}


In [ ]:
# ============================================================================
# API FUNCTIONS: List and retrieve inquiries
# ============================================================================

def list_inquiries(page_size: int = 100, page_after: Optional[str] = None, 
                   filter_status: Optional[str] = None, 
                   filter_template_id: Optional[str] = None) -> Dict[str, Any]:
    """List inquiries with optional filters.
    
    According to Persona API docs: https://docs.withpersona.com/2023-01-05/api-reference/inquiries/list-inquiries
    Filter by inquiry-template-id to get inquiries from a specific template.
    """
    url = f"{BASE_URL}/inquiries"
    
    params = {
        "page[size]": page_size,
    }
    
    if page_after:
        params["page[after]"] = page_after
    
    if filter_status:
        params["filter[status]"] = filter_status
    
    if filter_template_id:
        # Use inquiry-template-id filter (not template-id)
        params["filter[inquiry-template-id]"] = filter_template_id
    
    r = requests.get(url, headers=HEADERS, params=params, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()


def get_inquiry(inquiry_id: str) -> Dict[str, Any]:
    """Retrieve a single inquiry with full details."""
    from urllib.parse import quote
    encoded_id = quote(inquiry_id, safe='')
    
    url = f"{BASE_URL}/inquiries/{encoded_id}"
    params = {"include": "verifications"}
    
    r = requests.get(url, headers=HEADERS, params=params, timeout=REQUEST_TIMEOUT)
    
    if r.status_code == 404:
        raise ValueError(f"Inquiry {inquiry_id} not found (404).")
    
    r.raise_for_status()
    return r.json()


def list_templates() -> Dict[str, Any]:
    """List all inquiry templates to find ones with 'Tel Aviv' or 'Israel' in the name.
    
    According to Persona API docs: https://docs.withpersona.com/2023-01-05/api-reference/inquiry-templates/list-inquiry-templates
    """
    url = f"{BASE_URL}/inquiry-templates"
    
    params = {"page[size]": 100}
    
    r = requests.get(url, headers=HEADERS, params=params, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()


def extract_name_fields_from_inquiry_object(inquiry_obj: Dict[str, Any]) -> Dict[str, Optional[str]]:
    """Extract first and last name from a single inquiry object (from list response).
    
    Names are in inquiry.attributes.name-first / name-last
    """
    attributes = inquiry_obj.get("attributes", {})
    
    first_name = attributes.get("name-first") or attributes.get("name_first")
    last_name = attributes.get("name-last") or attributes.get("name_last")
    
    # Normalize: treat empty strings as None
    first_name = first_name if (first_name and str(first_name).strip()) else None
    last_name = last_name if (last_name and str(last_name).strip()) else None
    
    return {
        "first_name": first_name,
        "last_name": last_name,
        "has_first_name": first_name is not None,
        "has_last_name": last_name is not None,
        "has_both_names": (first_name is not None) and (last_name is not None),
        "missing_first": first_name is None,
        "missing_last": last_name is None,
        "missing_any": (first_name is None) or (last_name is None),
    }


def extract_name_fields(inquiry_data: Dict[str, Any]) -> Dict[str, Optional[str]]:
    """Extract first and last name from full inquiry data (from get_inquiry response).
    
    Names can be in:
    - inquiry.attributes.name-first / name-last
    - verification attributes (from included verifications)
    """
    first_name = None
    last_name = None
    
    # Check inquiry attributes
    data = inquiry_data.get("data", {})
    attributes = data.get("attributes", {})
    
    first_name = attributes.get("name-first") or attributes.get("name_first")
    last_name = attributes.get("name-last") or attributes.get("name_last")
    
    # If not found, check verifications in included data
    if not first_name or not last_name:
        included = inquiry_data.get("included", [])
        for item in included:
            if isinstance(item, dict):
                item_type = item.get("type", "")
                if item_type.startswith("verification"):
                    verif_attrs = item.get("attributes", {})
                    if not first_name:
                        first_name = verif_attrs.get("name-first") or verif_attrs.get("name_first")
                    if not last_name:
                        last_name = verif_attrs.get("name-last") or verif_attrs.get("name_last")
    
    # Normalize: treat empty strings as None
    first_name = first_name if (first_name and str(first_name).strip()) else None
    last_name = last_name if (last_name and str(last_name).strip()) else None
    
    return {
        "first_name": first_name,
        "last_name": last_name,
        "has_first_name": first_name is not None,
        "has_last_name": last_name is not None,
        "has_both_names": (first_name is not None) and (last_name is not None),
        "missing_first": first_name is None,
        "missing_last": last_name is None,
        "missing_any": (first_name is None) or (last_name is None),
    }


In [6]:
# ============================================================================
# STEP 1: Find templates with 'Tel Aviv' or 'Israel' in the name
# ============================================================================

print("Fetching all inquiry templates...")
all_templates = []

# Fetch all templates with pagination
page_after = None
page_num = 1

while True:
    try:
        params = {"page[size]": 100}
        if page_after:
            params["page[after]"] = page_after
        
        url = f"{BASE_URL}/inquiry-templates"
        r = requests.get(url, headers=HEADERS, params=params, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        templates_response = r.json()
        
        templates = templates_response.get("data", [])
        all_templates.extend(templates)
        
        print(f"  Page {page_num}: {len(templates)} templates (total so far: {len(all_templates)})")
        
        # Check for pagination
        links = templates_response.get("links", {})
        page_after = links.get("next")
        
        if not page_after:
            break
        
        page_num += 1
        time.sleep(SLEEP_S)
        
    except Exception as e:
        print(f"  ❌ Error fetching page {page_num}: {e}")
        break

print(f"\nFound {len(all_templates)} total templates\n")

# Filter templates with 'Tel Aviv' or 'Israel' in name
target_templates = []
for template in all_templates:
    template_name = template.get("attributes", {}).get("name", "").lower()
    template_id = template.get("id", "")
    
    if "tel aviv" in template_name or "israel" in template_name:
        target_templates.append({
            "id": template_id,
            "name": template.get("attributes", {}).get("name", ""),
        })

print(f"Found {len(target_templates)} templates with 'Tel Aviv' or 'Israel' in name:")
for t in target_templates:
    print(f"  - {t['name']} (ID: {t['id']})")

if len(target_templates) == 0:
    print("\n⚠️ No templates found. Please check template names.")
else:
    print(f"\n✅ Will analyze inquiries from {len(target_templates)} template(s)")


Fetching all inquiry templates...
  Page 1: 20 templates (total so far: 20)

Found 20 total templates

Found 7 templates with 'Tel Aviv' or 'Israel' in name:
  - [New Israel Template] Government ID without Selfie - DL + ID testing (ID: itmpl_9PLpLUxXrxtjGonHryEZC9YCDpuT)
  - [New Tel Aviv Template] Government ID and Selfie - age 18 (ID: itmpl_RnxdB4zi3FK42zpUVQYxsC7HyEms)
  -  [New Israel Template] Government ID without Selfie (ID: itmpl_rCjZgAvPrY1ickPyqrADozJNBsFE)
  - [Auto-Classification Testing] Tel Aviv Government ID and Selfie (ID: itmpl_dCtrHtiGaUMyTX7wEMmvi4CKcoLW)
  - [New Tel Aviv Template] Government ID and Selfie (ID: itmpl_bvrUboszYw1ywXK4PACDXZrQWqrE)
  - KYC: GovID (Bird Israel Except Tel Aviv) (ID: itmpl_yyzGfnMANkUrKi4ShAaN3xCa9FrP)
  - KYC: GovID + Selfie (Bird Israel > Tel Aviv) (ID: itmpl_t6hWr5umEqhgXu6uLFzmLu7hAfLi)

✅ Will analyze inquiries from 7 template(s)


In [12]:
# ============================================================================
# STEP 2: Fetch all inquiries from target templates (then filter by approved status)
# ============================================================================

# Initialize variable to avoid NameError
all_inquiries = []

if len(target_templates) == 0:
    print("⚠️ No target templates found. Skipping inquiry fetch.")
else:
    print(f"Fetching inquiries from {len(target_templates)} template(s)...")
    print("Note: We'll check all inquiries first to see status breakdown.\n")
    
    # First, let's test if we can fetch ANY inquiries at all (no filters)
    print("=" * 80)
    print("DEBUG: Testing API connection - fetching first 5 inquiries (any template, any status)...")
    print("=" * 80)
    try:
        test_response = list_inquiries(page_size=5, filter_status=None, filter_template_id=None)
        test_inquiries = test_response.get("data", [])
        print(f"✅ API connection works! Found {len(test_inquiries)} inquiries in test")
        if test_inquiries:
            test_inq = test_inquiries[0]
            test_attrs = test_inq.get("attributes", {})
            test_status = test_attrs.get("status", "unknown")
            test_template_rel = test_inq.get("relationships", {}).get("inquiry-template", {}).get("data", {})
            test_template_id = test_template_rel.get("id", "unknown")
            print(f"   Sample inquiry status: {test_status}")
            print(f"   Sample inquiry template ID: {test_template_id}")
    except Exception as e:
        print(f"❌ API test failed: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "=" * 80)
    print("Now fetching inquiries from target templates...")
    print("=" * 80 + "\n")
    
    for template in target_templates:
        template_id = template["id"]
        template_name = template["name"]
        
        print(f"\nTemplate: {template_name}")
        print(f"  Template ID: {template_id}")
        
        # First, try fetching ALL inquiries (any status) to see what we have
        print("  Fetching all inquiries (any status) for this template...")
        page_after = None
        page_num = 1
        total_fetched = 0
        status_counts = {}
        approved_inquiries = []
        
        while True:
            try:
                # Fetch without status filter
                print(f"    Making API call (page {page_num})...")
                response = list_inquiries(
                    page_size=100,
                    page_after=page_after,
                    filter_template_id=template_id,
                    # No status filter - get all
                )
                
                print(f"    ✅ API call successful")
                inquiries = response.get("data", [])
                print(f"    Received {len(inquiries)} inquiries in response")
                total_fetched += len(inquiries)
                
                for inquiry in inquiries:
                    inquiry_id = inquiry.get("id", "")
                    inquiry_attrs = inquiry.get("attributes", {})
                    status = inquiry_attrs.get("status", "unknown")
                    
                    # Count statuses
                    status_counts[status] = status_counts.get(status, 0) + 1
                    
                    # If approved, extract and add to our list
                    if status == "approved":
                        name_info = extract_name_fields_from_inquiry_object(inquiry)
                        
                        approved_inquiries.append({
                            "inquiry_id": inquiry_id,
                            "template_id": template_id,
                            "template_name": template_name,
                            "status": status,
                            "created_at": inquiry_attrs.get("created-at", ""),
                            **name_info,
                            "raw_data": inquiry,
                        })
                
                # Check for pagination
                links = response.get("links", {})
                page_after = links.get("next")
                
                if len(inquiries) > 0:
                    print(f"    Page {page_num}: {len(inquiries)} inquiries")
                
                if not page_after:
                    print(f"    No more pages")
                    break
                
                page_num += 1
                time.sleep(SLEEP_S)
                
            except Exception as e:
                print(f"    ❌ Error: {e}")
                import traceback
                traceback.print_exc()
                break
        
        print(f"  Total inquiries found: {total_fetched}")
        if status_counts:
            print(f"  Status breakdown: {status_counts}")
        print(f"  Approved inquiries: {len(approved_inquiries)}")
        
        all_inquiries.extend(approved_inquiries)
    
    print(f"\n✅ Total approved inquiries found across all templates: {len(all_inquiries)}")


Fetching inquiries from 7 template(s)...
Note: We'll check all inquiries first to see status breakdown.

DEBUG: Testing API connection - fetching first 5 inquiries (any template, any status)...
✅ API connection works! Found 5 inquiries in test
   Sample inquiry status: created
   Sample inquiry template ID: itmpl_1gUcnf5ePwvULFm9JSdSsbdv

Now fetching inquiries from target templates...


Template: [New Israel Template] Government ID without Selfie - DL + ID testing
  Template ID: itmpl_9PLpLUxXrxtjGonHryEZC9YCDpuT
  Fetching all inquiries (any status) for this template...
    Making API call (page 1)...
    ✅ API call successful
    Received 0 inquiries in response
    No more pages
  Total inquiries found: 0
  Approved inquiries: 0

Template: [New Tel Aviv Template] Government ID and Selfie - age 18
  Template ID: itmpl_RnxdB4zi3FK42zpUVQYxsC7HyEms
  Fetching all inquiries (any status) for this template...
    Making API call (page 1)...
    ✅ API call successful
    Received 0 inquir

In [10]:
# ============================================================================
# STEP 3: Verify names (names already extracted in Step 2, but we can verify)
# ============================================================================

# Initialize variable to avoid NameError
results = []

if len(all_inquiries) == 0:
    print("⚠️ No inquiries to analyze.")
else:
    print(f"Analyzing {len(all_inquiries)} approved inquiries...\n")
    
    # Names were already extracted in Step 2, so we can use them directly
    # Optionally, we can fetch full details for inquiries with missing names to check verifications
    results = []
    missing_count = 0
    
    for i, inquiry_info in enumerate(all_inquiries, start=1):
        inquiry_id = inquiry_info["inquiry_id"]
        has_both = inquiry_info.get("has_both_names", False)
        
        # If names are missing, try fetching full inquiry to check verifications
        if not has_both:
            missing_count += 1
            print(f"[{i}/{len(all_inquiries)}] {inquiry_id} (missing names, checking verifications)... ", end="", flush=True)
            try:
                # Fetch full inquiry with verifications
                full_inquiry = get_inquiry(inquiry_id)
                
                # Extract name fields from full inquiry (may find names in verifications)
                name_info = extract_name_fields(full_inquiry)
                
                # Update with verification data if found
                inquiry_info.update(name_info)
                
                status_icon = "✅" if name_info["has_both_names"] else "⚠️"
                print(f"{status_icon} First: {name_info['first_name'] or 'MISSING'}, Last: {name_info['last_name'] or 'MISSING'}")
                
                time.sleep(SLEEP_S)
            except Exception as e:
                print(f"❌ Error: {str(e)[:50]}")
        elif i <= 10 or i % 100 == 0:  # Show first 10 and then every 100th
            # Names already present, just show sample
            status_icon = "✅"
            first = inquiry_info.get("first_name", "MISSING")
            last = inquiry_info.get("last_name", "MISSING")
            print(f"[{i}/{len(all_inquiries)}] {status_icon} {first} {last}")
        
        results.append(inquiry_info)
    
    print(f"\n✅ Analysis complete!")
    if missing_count > 0:
        print(f"   Checked {missing_count} inquiries with missing names in verifications")


⚠️ No inquiries to analyze.


In [ ]:
# ============================================================================
# STEP 4: Summary and Results Table
# ============================================================================

# Initialize variable to avoid NameError (in case Cell 5 wasn't run)
try:
    _ = results
except NameError:
    results = []

if len(results) == 0:
    print("⚠️ No results to display.")
else:
    df = pd.DataFrame(results)
    
    # Remove raw_data column for display
    display_df = df.drop(columns=["raw_data"], errors="ignore")
    
    # Summary statistics
    total = len(df)
    missing_first = df["missing_first"].sum()
    missing_last = df["missing_last"].sum()
    missing_any = df["missing_any"].sum()
    has_both = df["has_both_names"].sum()
    
    print("=" * 80)
    print("SUMMARY: Approved Inquiries with Missing Names")
    print("=" * 80)
    print(f"Total approved inquiries analyzed: {total}")
    print(f"\nName Status:")
    print(f"  ✅ Has both first and last name: {has_both} ({has_both/total*100:.1f}%)")
    print(f"  ⚠️  Missing first name: {missing_first} ({missing_first/total*100:.1f}%)")
    print(f"  ⚠️  Missing last name: {missing_last} ({missing_last/total*100:.1f}%)")
    print(f"  ❌ Missing at least one name: {missing_any} ({missing_any/total*100:.1f}%)")
    print("=" * 80)
    
    # Breakdown by template
    if "template_name" in df.columns:
        print("\nBreakdown by Template:")
        template_summary = df.groupby("template_name").agg({
            "inquiry_id": "count",
            "missing_any": "sum",
            "missing_first": "sum",
            "missing_last": "sum",
        }).rename(columns={"inquiry_id": "total", "missing_any": "missing_any_count"})
        template_summary["missing_any_pct"] = (template_summary["missing_any_count"] / template_summary["total"] * 100).round(1)
        print(template_summary)
    
    # Display inquiries with missing names
    print("\n" + "=" * 80)
    print("Inquiries with Missing Names:")
    print("=" * 80)
    
    missing_df = df[df["missing_any"]].copy()
    if len(missing_df) > 0:
        display_cols = ["inquiry_id", "template_name", "first_name", "last_name", "missing_first", "missing_last", "created_at"]
        display_cols = [c for c in display_cols if c in missing_df.columns]
        display(missing_df[display_cols])
    else:
        print("✅ All inquiries have both first and last names!")
    
    # Save to CSV
    output_file = "approved_inquiries_name_analysis.csv"
    display_df.to_csv(output_file, index=False)
    print(f"\n✅ Results saved to: {output_file}")
